In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
english_df = pd.read_csv("/content/final_english_merge_train_data.csv")
hindi_df = pd.read_csv("/content/final_dataset_hindi.csv")
bengali_df = pd.read_csv("/content/merged_bengali_dataset.csv")
hindi_df2 = pd.read_csv("/content/english_to_hindi.csv")
bengali_df2 = pd.read_csv("/content/english_to_bengali.csv")

In [ ]:
print(english_df.head())

                                            sentence    label
0  : If we are going to slow climate change, we m...    FAVOR
1  World leaders duped by manipulated global warm...  AGAINST
2  To be sure the American greens have greater am...     NONE
3  Dealing with global warming is an internationa...    FAVOR
4  : Concussion is like climate change It isn't r...  AGAINST


In [ ]:
print(hindi_df.head())

                                                text
0  सूर्य ग्रहण: दिन में चार मिनट तक रहेगा अंधेरा,...
1  भारतीय महिला बैडमिंटन टीम ने यंग ब्रिगेड के सह...
2  नेपाल में बोधिचित्त वृक्षों की चिंता: 'सोने की...
3  आईपीएल के सुपर संडे में कई रिकॉर्ड बने, लेकिन ...
4  भारत ईरान ने मतभेद भुलाकर किया ये अहम समझौता, ...


In [ ]:
print(bengali_df.head())

                                                text
0  মার্কিন মহাকাশ সংস্থা নাসা আয়োজিত বিশ্বের বৃহ...
1  বঙ্গবন্ধুর জন্ম শতবর্ষ উপলক্ষে সারাদেশে ১ কোটি...
2  জার্মানির আন্তর্জাতিক সম্প্রচার কেন্দ্র ডয়চে ভ...
3  অর্থমন্ত্রী আ হ ম মুস্তফা কামাল বলেছেন, পেঁয়াজ...
4  কাপ্তাই হ্রদের দূষণ দূর করতে অভিযান পরিচালনা ক...


In [ ]:
print(hindi_df2.head())

                                            sentence    label  \
0  : If we are going to slow climate change, we m...    FAVOR   
1  World leaders duped by manipulated global warm...  AGAINST   
2  To be sure the American greens have greater am...     NONE   
3  Dealing with global warming is an internationa...    FAVOR   
4  : Concussion is like climate change It isn't r...  AGAINST   

                                               hindi  
0  : यदि हम जलवायु परिवर्तन को धीमा करना चाहते है...  
1  ग्लोबल वार्मिंग डेटा में हेरफेर करके विश्व नेत...  
2  यह सुनिश्चित करने के लिए कि मैक्रोन की तरह अमे...  
3  ग्लोबल वार्मिंग से निपटना एक अंतरराष्ट्रीय जिम...  
4  : हिलाना जलवायु परिवर्तन की तरह है यह वास्तविक...  


In [ ]:
print(bengali_df2.head())

                                            sentence    label  \
0  : If we are going to slow climate change, we m...    FAVOR   
1  World leaders duped by manipulated global warm...  AGAINST   
2  To be sure the American greens have greater am...     NONE   
3  Dealing with global warming is an internationa...    FAVOR   
4  : Concussion is like climate change It isn't r...  AGAINST   

                                             bengali  
0  : আমরা যদি জলবায়ু পরিবর্তনের গতি কমাতে যাচ্ছি...  
1  বিশ্বনেতারা বিশ্ব উষ্ণায়নের তথ্য ব্যবহার করে ...  
2  আমেরিকান সবুজ শাক-সবুজদের শুধু কার্বন ট্যাক্স ...  
3  বৈশ্বিক উষ্ণতা মোকাবেলা একটি আন্তর্জাতিক দায়িত্ব  
4  : উপহাস জলবায়ু পরিবর্তনের মতো এটি বাস্তব নয় ...  


In [ ]:
english = english_df["sentence"].astype(str)

hindi = hindi_df["text"].astype(str)

bengali = bengali_df["text"].astype(str)

hindi2 = hindi_df2["hindi"].astype(str)

bengali2 = bengali_df2["bengali"].astype(str)

In [ ]:
all_text = pd.concat(
    [
        english,
        hindi,
        bengali,
        hindi2,
        bengali2
    ],
    ignore_index=True
)

all_text = all_text.dropna()

all_text = all_text.drop_duplicates()

all_text = all_text.str.strip()

all_text = all_text[all_text!=""]

print("Total sentences:",len(all_text))

Total sentences: 46984


In [ ]:
mlm_df = pd.DataFrame()

mlm_df["text"] = all_text

In [ ]:
train_df, valid_df = train_test_split(
    mlm_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

In [ ]:
train_dataset = Dataset.from_pandas(train_df)

valid_dataset = Dataset.from_pandas(valid_df)

In [ ]:
MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [ ]:
MAX_LENGTH = 128

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_valid = valid_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/42285 [00:00<?, ? examples/s]

Map:   0%|          | 0/4699 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google/muril-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

Mounted at /content/drive


In [ ]:
from transformers import EarlyStoppingCallback

In [ ]:
training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/MURIL_MLM6",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_ratio=0.06,

    fp16=True,

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    save_total_limit=1,

    report_to="none"
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.840093,2.187337
2,4.433313,2.113430
3,4.101354,2.017991
4,4.105219,2.013388
5,4.089751,1.948824


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=13215, training_loss=4.332267174664835, metrics={'train_runtime': 7835.4709, 'train_samples_per_second': 26.983, 'train_steps_per_second': 1.687, 'total_flos': 1.39390974286272e+16, 'train_loss': 4.332267174664835, 'epoch': 5.0})

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/Best_MURIL_MLM6"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Best_MURIL_MLM6/tokenizer_config.json',
 '/content/drive/MyDrive/Best_MURIL_MLM6/tokenizer.json')